# Recitation 1 — Sorting Algorithms (Python)

COMPSCI 301 · Algorithms & Databases — Duke Kunshan University

Practice notebook for [Recitation 1](https://harryc919.github.io/TA-CS301/recitation1.html).

## How to use this notebook

- This notebook mirrors the [Recitation 1 page](https://harryc919.github.io/TA-CS301/recitation1.html): the same concept notes and the same six problems. Reference solutions in all six languages stay on the website — try each problem before reading them.
- **Setup:** follow [Recitation 0](https://harryc919.github.io/TA-CS301/recitation0.html) once to create your `cs301-notebooks` uv project. Save this notebook into that folder and open it with the **Python (CS301)** kernel.
- **Work top to bottom.** Read the notes, then for each problem: read the brief, complete the starter-template cell, and run the self-check cell below it with Shift+Enter.
- A self-check cell fails with an `AssertionError` while the function is still the untouched stub. That is expected — implement, re-run the template cell, then re-run the check.
- Keep the function signature unchanged, sort manually (no `sorted(...)` or `list.sort(...)` in your implementation), and mind the "must not modify the input" requirements where stated.
- Nothing here is submitted or graded. Bring your attempt, or the exact line that breaks, to the recitation session.

## Five things to remember

1. No comparison sort beats **Ω(n log n)** comparisons in the worst case: n items come in n! orders, and every order must be told apart.
2. **Bubble and insertion sort use O(1) extra space**; quick sort partitions in place but also needs a recursion stack. Standard array merge sort uses O(n) extra space.
3. Insertion sort's **shift count equals the number of inversions**, I. Its total time is Θ(n + I), so sorted input still takes Θ(n). Problem 2 builds on this.
4. **Stability** means equal keys keep their original order, which matters as soon as records carry several fields.
5. A **random pivot plus 3-way partition** avoids predictable bad splits on sorted input and skips equal keys. Expected time is O(n log n); the O(n²) worst case remains.

## Learning goals

By the end of this pass you should be able to:

- state what a comparison-based sort is and why Ω(n log n) bounds it;
- trace bubble, insertion, merge and quick sort on a small array by hand;
- read best, average and worst cases, keeping time separate from extra space;
- define stable, in-place and adaptive, and label the four sorts with them;
- choose a sort for a stated situation, including the hybrids libraries ship.

## The big picture

Sorting is the warm-up problem of algorithm analysis: trivial to state, easy to implement badly, and bounded below by Ω(n log n) once only comparisons are allowed.

### What sorting is for

An ordered array is easier to work with everywhere. Binary search needs it, deduplication becomes one scan, and grouping, medians and the greedy or dynamic-programming algorithms of later weeks all start from an order. Databases do the same work with ORDER BY, sort-merge joins and B-tree indexes.

### What "comparison-based" means

All four sorts learn about two elements only by comparing them: less than, equal, greater than. They never inspect digits, bytes or hashes, so the same code sorts integers, strings or whole records whenever a total order exists. That restriction is exactly what makes the lower bound apply; counting sort and radix sort escape it by looking at values.

### Why Ω(n log n) is a lower bound

With n distinct values there are n! possible input orders, and the algorithm must tell all of them apart, so its decision tree needs at least n! leaves. A binary tree with L leaves has height at least log₂ L, so some input forces at least log₂(n!) comparisons. Since log₂(n!) ≈ n log₂ n − 1.44n, that is Ω(n log n). Merge sort matches the bound; quick sort matches it on average.

### In place, stable, adaptive

Strictly in place: O(1) extra memory including the call stack. Quick sort's partition is in place, but its recursion stack uses additional memory; the tables below exclude that stack when labeling its data rearrangement as in place. Stable: equal keys keep their relative order. Adaptive: already or nearly sorted input makes the algorithm clearly faster. Table 1 applies all three to the four sorts.

### Why quadratic sorts still matter

For n = 10, insertion sort needs at most 45 shifts. Its low overhead can make it useful for small inputs, and its running time on nearly sorted data approaches O(n). Practical hybrid sorts exploit this: Timsort extends short runs with insertion sort; common introsort implementations use insertion sort on small partitions and heapsort if recursion grows too deep. The useful cutoff depends on the implementation and data.

## 1. Bubble sort

*Average O(n²) · Worst O(n²) · Space O(1) · Stable, in place*

### The idea

Bubble sort makes repeated passes, comparing neighbours and swapping them when they are out of order. Each pass floats the largest remaining element to the end of the unsorted region, so after k passes the last k elements are final and the next pass can be shorter. That growing sorted tail is the picture worth remembering.

### How it works

1. Compare the first two elements and swap them if the left one is larger, then move one position right and repeat across the unsorted region.
2. The largest remaining element is now at the region's right end; shrink the region by one.
3. Set a flag on every swap. A full pass without swaps means the array is sorted, so stop early.
4. Repeat until a pass is clean or one element is left.

### Worked example

Bubble sort on `[5, 2, 4, 1, 3]`:

*Tracing bubble sort on [5, 2, 4, 1, 3]. The sorted tail grows by one per pass.*

| Pass | Array after the pass | What the pass settled |
| --- | --- | --- |
| start | `[5, 2, 4, 1, 3]` | nothing yet |
| 1 | `[2, 4, 1, 3, 5]` | 5 reaches the end |
| 2 | `[2, 1, 3, 4, 5]` | 4 joins the sorted tail |
| 3 | `[1, 2, 3, 4, 5]` | the array becomes sorted |
| 4 | `[1, 2, 3, 4, 5]` | no swaps, so stop |

> Passes 1 to 3 make 4, 3 and 2 comparisons; pass 4 makes 1 and then stops early.

### Pseudocode

**bubble_sort(a)** (pseudocode)

```
for pass = 1 to n - 1:
  swapped = false
  for i = 0 to n - pass - 1:
    if a[i] > a[i + 1]:
      swap a[i] and a[i + 1]
      swapped = true
  if not swapped:
    break                    # array is already sorted: stop
return a
```

### Complexity

- **Best case** — O(n): sorted input, one pass of n − 1 comparisons, no swaps, early exit.
- **Average and worst** — O(n²): about n(n − 1)/2 comparisons; reverse-sorted input swaps at every comparison.
- **Extra space** — O(1): a temporary and two counters.
- **Stability** — Stable: only strictly greater neighbours swap, so equal values never cross.

> **Watch out.** Without the early-exit check, this bubble-sort loop makes Θ(n²) comparisons even on sorted input. Reset the flag before every pass; otherwise a swap in an earlier pass can prevent a later clean pass from stopping the loop. Bubble sort is a teaching tool for tracing, not a sort to ship.

## 2. Insertion sort

*Best O(n) · Average O(n²) · Worst O(n²) · Space O(1) · Stable, in place, adaptive*

### The idea

Insertion sort grows a sorted prefix one element at a time, the way most people sort a hand of cards: take the next card and slide it left until it sits in order. Everything before the current position is already sorted, and the new element is inserted into that prefix.

### How it works

1. Start with the first element as a sorted prefix of length one.
2. Take the next element as the key and scan left through the prefix.
3. Shift every element greater than the key one position right; one shift moves one element exactly one place.
4. Stop at an element ≤ the key or at the start of the array, and put the key in the gap.
5. Repeat until the prefix spans the array.

### Worked example

Insertion sort on `[5, 2, 4, 1, 3]`, sorted prefix in bold:

*Tracing insertion sort on [5, 2, 4, 1, 3]. Seven shifts in total = seven inversions.*

| Insert | Array after inserting | Shifts used |
| --- | --- | --- |
| start | `[5 \| 2, 4, 1, 3]` | 0 |
| 2 | `[2, 5] \| 4, 1, 3` | 1 |
| 4 | `[2, 4, 5] \| 1, 3` | 1 |
| 1 | `[1, 2, 4, 5] \| 3` | 3 |
| 3 | `[1, 2, 3, 4, 5]` | 2 |

> The shifts add up: 1 + 1 + 3 + 2 = 7, and the array has exactly 7 inversions. An inversion is a pair i < j with a[i] > a[j]. Each shift lets the key pass one larger predecessor, accounting for exactly one original inversion. Counting all such shifts counts each inversion once. Problem 2 asks you to implement this count.

### Pseudocode

**insertion_sort(a)** (pseudocode)

```
for i = 1 to n - 1:
  key = a[i]
  j = i - 1
  while j >= 0 and a[j] > key:
    a[j + 1] = a[j]          # shift one position right
    j = j - 1
  a[j + 1] = key             # drop the key into the gap
return a
```

### Complexity

- **Best case** — O(n): sorted input, n − 1 comparisons, no shifts.
- **Average and worst** — O(n²): reverse-sorted input needs n(n − 1)/2 shifts, about 12.5 million moves for n = 5,000; random input averages roughly n²/4.
- **Extra space** — O(1): the key and two indices.
- **Stability and adaptivity** — Stable, and adaptive: exactly I shifts for I inversions in the original array, plus Θ(n) loop overhead, giving Θ(n + I) total time.

> **Watch out.** Binary-searching the insertion point only trades comparisons for shifts. The shifts still take Θ(n²) moves in the worst case and dominate on arrays; cutting them needs a different data structure.

## 3. Merge sort

*Best O(n log n) · Average O(n log n) · Worst O(n log n) · Space O(n) · Stable, not in place*

### The idea

Merge sort splits the array in half, sorts each half recursively, then merges the two sorted halves. Merging is where the work happens: compare the fronts, take the smaller element, and repeat. Each half arrives sorted, so every element is touched once per recursion level, and there are about log₂ n levels.

### How it works

1. A segment of length 0 or 1 is sorted: return it.
2. Split the segment at its midpoint and sort both halves recursively.
3. Merge: repeatedly append the smaller front element; on a tie take the left one, which keeps the sort stable.
4. Append whatever is left over and copy the merged run back into place.

### Worked example

Merge sort on `[5, 2, 4, 1, 3]`, using the same floor midpoint as the pseudocode:

*Tracing merge sort on [5, 2, 4, 1, 3]. Depth is ⌈log2 5⌉ = 3 levels.*

| Stage | Arrays | Result |
| --- | --- | --- |
| split | `[5, 2]` and `[4, 1, 3]` | left has 2 elements; right has 3 |
| left merge | `[5]` + `[2]` | `[2, 5]` |
| right inner merge | `[1]` + `[3]` | `[1, 3]` |
| right merge | `[4]` + `[1, 3]` | `[1, 3, 4]` |
| final merge | `[2, 5]` + `[1, 3, 4]` | `[1, 2, 3, 4, 5]` |

> Five elements give ⌈log₂ 5⌉ = 3 levels, and the total merge work per level is at most proportional to five elements (some branches reach singletons earlier): that is the O(n log n) argument.

### Pseudocode

**merge_sort(a)** (pseudocode)

```
merge_sort(a):
  if length(a) <= 1:
    return a
  mid   = floor(length(a) / 2)
  left  = merge_sort(a[0 .. mid - 1])
  right = merge_sort(a[mid .. length(a) - 1])
  return merge(left, right)

merge(left, right):
  result = []
  i = 0
  j = 0
  while i < length(left) and j < length(right):
    if left[i] <= right[j]:   # <= keeps equal keys in order
      append left[i] to result
      i = i + 1
    else:
      append right[j] to result
      j = j + 1
  append left[i .. end] and right[j .. end] to result
  return result
```

### Complexity

- **Best, average, worst** — All O(n log n): log₂ n levels with O(n) merging per level, whatever the input order.
- **Extra space** — O(n) for the merge buffer. Linked lists can be merged by relinking nodes instead, which is why merge sort is the standard list sort.
- **Stability** — Yes, if the merge takes from the left half on ties.
- **In place** — No: the buffer is the price of the guaranteed bound.

> **Watch out.** The tie-break is the stability switch: writing `<` instead of `<=` flips equal keys across halves. Standard merge sort has Θ(n log n) time in the best, average and worst cases; the exact comparison counts can differ. This array version uses O(n) extra memory. Use indices when merging: removing the first element of a Python list takes linear time.

## 4. Quick sort

*Average O(n log n) · Worst O(n²) · Space O(log n) average · Not stable, in-place partition*

### The idea

Quick sort picks a pivot, partitions the segment so smaller elements lie left and larger right, then sorts both sides recursively. Unlike merge sort, the hard part comes first: after partitioning, the pivot sits in its final position and never moves again.

### How it works

1. Pick a pivot. The trace below uses the last element; the exercises want a random index.
2. Partition: swap every element smaller than the pivot into a growing left region, then swap the pivot into the boundary.
3. The pivot is now final: everything left of it is smaller, everything right larger or equal.
4. Recurse on both sides. A 3-way partition groups equal keys in the middle and keeps them out of the recursion.
5. Stop at segments of zero or one element.

### Worked example

Quick sort on `[8, 3, 5, 1, 9, 2]`, last element as pivot (a simplified illustration; the exercise requires a random pivot):

*Tracing quick sort with last-element pivots: pivot 2, then 3, then 5, then 8.*

| Partition | Segment before | Pivot | Array after partitioning |
| --- | --- | --- | --- |
| 1 | `[8, 3, 5, 1, 9, 2]` | 2 | `[1, 2, 5, 8, 9, 3]` |
| 2 | `[5, 8, 9, 3]` | 3 | `[1, 2, 3, 8, 9, 5]` |
| 3 | `[8, 9, 5]` | 5 | `[1, 2, 3, 5, 9, 8]` |
| 4 | `[9, 8]` | 8 | `[1, 2, 3, 5, 8, 9]` |
| done | singletons left | — | `[1, 2, 3, 5, 8, 9]` |

> Each partition is one pass over its segment. The first split is uneven: one element lies left of the pivot and four lie right. Each later split has an empty left side, so this trace also illustrates poor balance. On sorted input, last-element pivots leave subproblems of sizes 0 and n − 1, giving (n − 1) + (n − 2) + ... + 1 comparisons. Random pivots prevent sorted order alone from forcing these bad splits; they do not eliminate the O(n²) worst case.

### Pseudocode

**quick_sort(a, lo, hi)** (pseudocode)

```
quick_sort(a, lo, hi):
  if lo >= hi:
    return
  p = partition(a, lo, hi)   # p is the pivot's final index
  quick_sort(a, lo, p - 1)
  quick_sort(a, p + 1, hi)

partition(a, lo, hi):
  pivot = a[hi]              # illustration: use a random index in practice
  i = lo
  for j = lo to hi - 1:
    if a[j] < pivot:
      swap a[i] and a[j]
      i = i + 1
  swap a[i] and a[hi]        # the pivot lands at index i
  return i
```

### Complexity

- **Best and average** — O(n log n): balanced partitions give log₂ n levels of O(n) work; random pivots give expected O(n log n) time on distinct keys. For duplicate-heavy input, use the 3-way partition required in P5.
- **Worst case** — O(n²): maximally unbalanced partitions, as first or last pivots cause on sorted input, or the illustrated Lomuto 2-way partition on all-equal input.
- **Extra space** — O(log n) average recursion stack, O(n) worst case; the partitioning itself is in place.
- **Stability** — Not stable: swaps move equal keys past each other.

> **Watch out.** The illustrated last-pivot Lomuto partition is quadratic on sorted or all-equal input. Use a random index plus a 3-way partition, which sets equal keys aside in one pass. Randomization gives an expected bound, not a worst-case guarantee; 3-way partition handles an all-equal segment in Θ(n). Quick sort is still not stable.

## Stability, in place, and the words we use

Three adjectives appear in every sorting discussion. They are independent, and the four sorts cover most useful combinations.

### Stable

A sort is stable when equal keys keep their original relative order. Take three student records — **(Alice, 90)**, **(Bob, 85)**, **(Cara, 90)** — and sort them by score ascending. Alice and Cara tie at 90, so the score alone does not decide between them. A stable sort returns Bob 85, Alice 90, Cara 90 and keeps the tie in place; an unstable sort may return Bob 85, Cara 90, Alice 90. That is fine when only scores matter, and wrong when the records were in name order. Sorting by a second field after a first only works when the second sort is stable.

### In place

Strictly in place means O(1) extra memory, including the call stack. Quick sort is often called in place because its partition rearranges the original array with O(1) workspace; its recursive version still uses an expected O(log n) stack and O(n) stack in the worst case. The tables use this conventional label with the stack shown separately. Standard array merge sort uses an O(n) buffer. P5 returns a copy, so its complete function uses O(n) space even though the sorting inside that copy partitions in place.

### Adaptive

A sort is adaptive when it is clearly faster on partly ordered data. Insertion sort is the model case: its shift count is I, the number of inversions, and total time is Θ(n + I). Bubble sort is adaptive only with the early-exit flag. Merge and quick sort are not, because their split or partition pattern ignores the order already present.

*Table 1. Which of the four sorts are stable, in place and adaptive.*

| Sort | Stable | In place | Adaptive |
| --- | --- | --- | --- |
| Bubble | Yes | Yes | Yes, with early exit |
| Insertion | Yes | Yes | Yes |
| Merge | Yes | No (O(n) buffer) | No |
| Quick | No | Yes, excluding stack | No |

> The buffer is what buys merge sort stability and a guaranteed bound; quick sort trades those for speed and locality.

## All four side by side

The pills from each section, collected in one table. Here n is the number of elements. The quick-sort row describes the 2-way version above on distinct keys; its average assumes random pivots. P5 uses 3-way partition, whose all-equal best case is Θ(n). The in-place column excludes the recursion stack, which is counted under extra space.

*Table 2. Bubble, insertion, merge and quick sort side by side.*

| Algorithm | Best | Average | Worst | Extra space | Stable | In-place | Notes |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Bubble | O(n) | O(n²) | O(n²) | O(1) | Yes | Yes | O(n) best case only with the early-exit flag |
| Insertion | O(n) | O(n²) | O(n²) | O(1) | Yes | Yes | Adaptive; shifts equal inversions |
| Merge | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes | No | Guaranteed bound; linked lists and external sorting |
| Quick | O(n log n) | O(n log n) | O(n²) | O(log n) avg | No | Yes | 2-way trace above; P5 adds 3-way partition |

> Constants matter as well as notation. Insertion sort can be faster on small arrays; the crossover depends on the implementation, element type and input. Practical libraries therefore often combine several algorithms.

## When to use which

### Small or nearly sorted

Insertion sort is a useful choice for small or nearly sorted arrays. Its time is Θ(n + I), where I is the number of inversions; the useful size cutoff depends on the implementation.

### Guaranteed worst case, stable

Use merge sort when O(n) memory is acceptable. It is also the natural choice for linked lists and external sorting, where sequential passes are all the disk gives you.

### General-purpose arrays

Quick sort with a random pivot and a 3-way partition is often efficient on arrays. This recursive version uses expected O(log n) stack space, O(n) in the worst case.

### Teaching only

Bubble sort, for tracing by hand and counting swaps in a demonstration. It is rarely a practical first choice.

### Bounded integer keys

Use counting sort or radix sort. They are not comparison-based, so they can beat the Ω(n log n) bound for special data; they come later in the course.

Real libraries ship hybrids: Timsort merges runs that insertion sort extends, and introsort mixes quick sort, heapsort and insertion sort. Library choices depend on the language, element type and implementation; do not assume one algorithm or one performance ranking applies everywhere.

## A preview: order statistics

**Optional exploration — not covered in class.** If only one order statistic is needed, sorting the whole array does extra work. For the k-th largest value (1-based k), the target index in ascending order is n − k. Quickselect partitions and continues only in the region containing that target. With 3-way partition, stop if the target is in the equal band.

Uniform random pivots give expected O(n) time, with O(n²) worst case. The retained region does not necessarily halve on each step. One way to see the expected bound: a pivot in the middle half occurs with probability at least 1/2 and leaves at most about 3/4 of the elements; only a constant expected number of partitions is needed for each such shrinkage. The work then forms a geometric sum. This is an intuition for the expected bound, not a guarantee that every partition is balanced.

An iterative implementation needs O(1) selection workspace and no recursion stack. The reference answers copy the input except in C, so those complete functions use O(n) extra space. Median-of-medians is a different pivot rule that can guarantee O(n) worst-case time.

Problem 6 on the exercises page builds directly on quick sort's partition.

## Before you start the exercises

The exercises page has six problems mapped to the four sorts. Use whichever language is fastest for you: C, C++, C#, Java, Python and JavaScript are all fine. Reference solutions are provided, but only after a real attempt; nothing is submitted.

- Trace the four worked examples above by hand before writing any code.
- Pick one language and stay with it. Porting a finished solution is cheap; switching mid-problem is not.
- Open the reference solution only after your code runs, or after you can name the exact line where you are stuck.
- Bring your attempt, or the exact place it breaks, to the practice session.

## Exercises

Six problems mapped to the four sorts above. Each problem has a brief (statement,
examples, constraints, signature, hidden hints), a starter-template cell, and a
self-check cell. Keep the signature unchanged and implement the algorithms
yourself — built-in sorts are off limits. Problem 6 is optional exploration and
is not covered in class.

## Problem 1 — Bubble Sort with Early Exit

**Difficulty:** Easy · **Topics:** Bubble sort · Early exit · In place

Given an integer array `nums`, sort it **in place** in ascending order using bubble sort.

Your implementation must include the early-exit optimization: track whether any swap happened during a pass. If a full pass makes no swaps, the array is already sorted, so stop immediately.

### Examples

**Example 1**
- Input: `nums = [5, 2, 4, 1, 3]`
- Output: `nums = [1, 2, 3, 4, 5]`
- Explanation: After the call the array is sorted in place; the function returns nothing.

**Example 2**
- Input: `nums = [1, 2, 3, 4]`
- Output: `nums = [1, 2, 3, 4]`
- Explanation: Already sorted: the first pass makes no swaps, so the loop exits after one pass. This is the O(n) best case.

### Constraints

- 1 ≤ `nums.length` ≤ 1000
- -10⁴ ≤ `nums[i]` ≤ 10⁴
- Sort in place, in ascending order. The function returns nothing.
- Do not use a built-in sort.

### Function signature (Python)

```python
def bubble_sort(nums: list[int]) -> None
```

The array is sorted in place in all six languages — the caller observes the result through the input array.

### Hints

<details><summary>Show hints (open only if you are stuck)</summary>
<ol>
<li>Compare adjacent pairs: for each pass, walk <code>j</code> from <code>0</code> to <code>n - 1 - pass</code> and swap <code>nums[j]</code> with <code>nums[j+1]</code> whenever they are out of order.</li>
<li>Reset an <code>swapped</code> flag to <code>false</code> at the start of every pass, set it to <code>true</code> whenever you swap, and stop as soon as a pass finishes with no swaps.</li>
<li>An already-sorted array must take a single pass. If your loop always runs the full n&nbsp;&minus;&nbsp;1 passes, the early exit is not wired up correctly.</li>
</ol>
</details>

In [ ]:
# TODO: implement bubble sort with the early-exit optimization.
def bubble_sort(nums: list[int]) -> None:
    pass


In [ ]:
# Self-check — fails while bubble_sort is still the untouched stub.
nums = [5, 2, 4, 1, 3]
bubble_sort(nums)
assert nums == [1, 2, 3, 4, 5], f"expected [1, 2, 3, 4, 5], got {nums}"

nums = [1, 2, 3, 4]
bubble_sort(nums)
assert nums == [1, 2, 3, 4], f"expected no changes, got {nums}"

print("Problem 1 checks passed.")

## Problem 2 — Insertion Sort: Count the Shifts

**Difficulty:** Easy · **Topics:** Insertion sort · Adaptive sorting · Counting

Sort `nums` **in place** in ascending order using insertion sort, then return the total number of **shifts** performed.

A shift is one element moving one position to the right to make room for the key. In the inner loop, each element you pass over and move right counts as one shift. Compare with strict `>` so that equal elements are never shifted.

The total number of shifts is exactly the number of inversions in the array (each shift accounts for one original inversion).

### Examples

**Example 1**
- Input: `nums = [5, 2, 4, 1]`
- Output: `return 5 (and nums = [1, 2, 4, 5])`
- Explanation: Insert 2: 5 shifts once (1). Insert 4: 5 shifts once (2). Insert 1: 5, 4, and 2 each shift once (5 in total).

**Example 2**
- Input: `nums = [1, 2, 3]`
- Output: `return 0`
- Explanation: Already sorted: nothing ever shifts.

**Example 3**
- Input: `nums = [2, 2, 2]`
- Output: `return 0`
- Explanation: Equal keys do not count as out of order, so nothing shifts.

### Constraints

- 1 ≤ `nums.length` ≤ 1000
- -10⁴ ≤ `nums[i]` ≤ 10⁴
- The shift count fits in a 32-bit integer.
- Sort in place. Do not use a built-in sort.

### Function signature (Python)

```python
def insertion_sort(nums: list[int]) -> int
```

The return value is the shift count; the sorted order is observed through the modified input array.

### Hints

<details><summary>Show hints (open only if you are stuck)</summary>
<ol>
<li>Keep a sorted prefix <code>nums[0..i-1]</code>. For each <code>nums[i]</code>, save it as <code>key</code>, shift every larger element one step right, then put <code>key</code> into the hole.</li>
<li>Count one shift per element moved right &mdash; that is, inside the inner loop, not once per insertion.</li>
<li>Use <code>nums[j-1] &gt; key</code>, not <code>&gt;=</code>. With <code>&gt;=</code>, Example 3 (<code>[2,2,2]</code>) performs 3 shifts instead of 0.</li>
</ol>
</details>

In [ ]:
# TODO: implement insertion sort and return the total number of shifts.
def insertion_sort(nums: list[int]) -> int:
    pass


In [ ]:
# Self-check — fails while insertion_sort is still the untouched stub.
nums = [5, 2, 4, 1]
shifts = insertion_sort(nums)
assert shifts == 5, f"expected 5 shifts, got {shifts}"
assert nums == [1, 2, 4, 5], f"expected [1, 2, 4, 5], got {nums}"

assert insertion_sort([1, 2, 3]) == 0

nums = [2, 2, 2]
assert insertion_sort(nums) == 0, "equal keys must not shift (use >, not >=)"

print("Problem 2 checks passed.")

## Problem 3 — Stable Sort by Word Length

**Difficulty:** Easy&ndash;Medium · **Topics:** Stability · Insertion sort · Merge sort

Given an array of strings `words`, return a new array containing the words sorted by length in ascending order.

Words of equal length must keep their original relative order — the sort must be **stable**. Implement the sorting yourself; built-in sort functions are not allowed.

Insertion sort stays stable if you shift only elements strictly greater than the key; merge sort stays stable if the merge takes from the left half on a tie. The usual in-place quick sort is not stable, so do not use that version here. In C the function sorts the given array of pointers in place; in the other five languages it returns a sorted array.

### Examples

**Example 1**
- Input: `words = ["pear", "kiwi", "apple", "fig", "banana"]`
- Output: `["fig", "pear", "kiwi", "apple", "banana"]`
- Explanation: Lengths are 3, 4, 5, 6. “pear” and “kiwi” are both length 4, and “pear” came first, so it stays first.

**Example 2**
- Input: `words = ["bb", "aa", "cc"]`
- Output: `["bb", "aa", "cc"]`
- Explanation: All words have length 2, so a stable sort changes nothing.

**Example 3**
- Input: `words = ["ccc", "a", "bb", "ab", "ba"]`
- Output: `["a", "bb", "ab", "ba", "ccc"]`
- Explanation: “ab” and “ba” are both length 2 and keep their original order.

### Constraints

- 1 ≤ `words.length` ≤ 1000
- Each word is 1–100 lowercase English letters.
- The sort must be stable. Do not use a built-in sort.

### Function signature (Python)

```python
def sort_by_length(words: list[str]) -> list[str]
```

C sorts the pointer array in place and returns nothing; the other five languages return the sorted array.

### Hints

<details><summary>Show hints (open only if you are stuck)</summary>
<ol>
<li>Insertion sort: move each word left past every word that is <em>longer</em>, stopping at a shorter or equal-length neighbour. Equal-length words therefore never jump over each other.</li>
<li>Merge sort: when the two fronts have equal length, take the one from the left half. That single <code>&lt;=</code> comparison is what makes the merge stable.</li>
<li>Test Example 3 carefully: the three length-2 words must stay in their original order, so compare lengths only and never the strings themselves.</li>
</ol>
</details>

In [ ]:
# TODO: implement a stable sort by length (return a new sorted list).
def sort_by_length(words: list[str]) -> list[str]:
    pass


In [ ]:
# Self-check — fails while sort_by_length is still the untouched stub.
words = ["pear", "kiwi", "apple", "fig", "banana"]
result = sort_by_length(words)
assert result == ["fig", "pear", "kiwi", "apple", "banana"], f"got {result}"

assert sort_by_length(["bb", "aa", "cc"]) == ["bb", "aa", "cc"], (
    "equal lengths must keep their original order")

assert sort_by_length(["ccc", "a", "bb", "ab", "ba"]) == ["a", "bb", "ab", "ba", "ccc"], (
    "the three length-2 words must keep their original order")

print("Problem 3 checks passed.")

## Problem 4 — Count Inversions

**Difficulty:** Medium · **Topics:** Merge sort · Counting · 64-bit integers

An **inversion** is a pair of indices `(i, j)` with `i < j` and `nums[i] > nums[j]`. Return the total number of inversions in `nums`.

A brute-force double loop is O(n²), which is too slow for the largest tests. The expected solution modifies merge sort to count inversions while merging, in O(n log n).

The answer can reach 4,999,950,000. Use `long long` in C/C++, `long` in Java/C#, or Python's unbounded `int`. JavaScript `Number` represents every possible answer here exactly. The function must not destroy the caller's array.

### Examples

**Example 1**
- Input: `nums = [8, 4, 2, 1]`
- Output: `return 6`
- Explanation: Every one of the six pairs is out of order.

**Example 2**
- Input: `nums = [1, 20, 6, 4, 5]`
- Output: `return 5`
- Explanation: The inversions are (20,6), (20,4), (20,5), (6,4) and (6,5).

**Example 3**
- Input: `nums = [2, 2, 2]`
- Output: `return 0`
- Explanation: Equal values are not inversions: the comparison is strict.

### Constraints

- 1 ≤ `nums.length` ≤ 100,000
- -10⁹ ≤ `nums[i]` ≤ 10⁹
- Use a type that represents 4,999,950,000 exactly (64-bit integer, Python `int`, or JavaScript `Number`).
- Do not modify the input array. Do not use a built-in sort in the solution.

### Function signature (Python)

```python
def count_inversions(nums: list[int]) -> int
```

C++ receives the array as a `const` reference — copy it internally before sorting. C receives a `const` pointer.

### Hints

<details><summary>Show hints (open only if you are stuck)</summary>
<ol>
<li>Recap merge sort: recursively sort the two halves, then merge. The counting happens during the merge, not afterwards.</li>
<li>When the right front is strictly smaller than the left front and you take it, every element still unmerged in the left half is larger than it. That is <code>left.length - i</code> new inversions.</li>
<li>Trust but verify: on <code>[4,3,2,1]</code> merging <code>[4]</code> with <code>[3]</code> adds 1, merging <code>[2]</code> with <code>[1]</code> adds 1, and the final merge of <code>[3,4]</code> with <code>[1,2]</code> adds 4 &mdash; 6 in total.</li>
</ol>
</details>

In [ ]:
# TODO: implement count inversions by extending merge sort.
def count_inversions(nums: list[int]) -> int:
    pass


In [ ]:
# Self-check — fails while count_inversions is still the untouched stub.
nums = [8, 4, 2, 1]
result = count_inversions(nums)
assert result == 6, f"expected 6, got {result}"
assert nums == [8, 4, 2, 1], "the input array must not be modified"

assert count_inversions([1, 20, 6, 4, 5]) == 5
assert count_inversions([2, 2, 2]) == 0, "equal values are not inversions (strict >)"

# Requires the O(n log n) merge-sort approach; also proves the 64-bit range.
assert count_inversions(list(range(100_000, 0, -1))) == 4_999_950_000

print("Problem 4 checks passed.")

## Problem 5 — Quick Sort: Random Pivot, 3-Way Partition

**Difficulty:** Medium · **Topics:** Quick sort · Randomization · Duplicates

Given an integer array `nums`, return a **new array** with the same values sorted in ascending order, implemented as quick sort.

Two requirements. First, choose the pivot **uniformly at random** (pick a random index in the current segment and use its value as the pivot). Second, use a **3-way partition** (the Dutch national flag algorithm): after partitioning, all values equal to the pivot are already in their final positions, so you recurse only on the strictly-smaller and strictly-larger regions.

The strict-less-than Lomuto 2-way partition in the notes degrades to O(n²) on an array where every value is equal. With random pivots, the 3-way version has expected O(n log n) time and processes an all-equal array in O(n); its worst case is still O(n²). Do not modify the input array.

### Examples

**Example 1**
- Input: `nums = [5, 2, 4, 1, 3]`
- Output: `[1, 2, 3, 4, 5]`
- Explanation: The input array itself is unchanged.

**Example 2**
- Input: `nums = [3, 1, 3, 3, 2, 3]`
- Output: `[1, 2, 3, 3, 3, 3]`
- Explanation: All four 3s are grouped by the partition and never examined again: that is why duplicates are cheap here.

### Constraints

- 0 ≤ `nums.length` ≤ 100,000
- -10⁶ ≤ `nums[i]` ≤ 10⁶
- The tests include many duplicates, fully sorted inputs, and an empty array.
- Do not modify the input array. Do not use a built-in sort.

### Function signature (Python)

```python
def quick_sort(nums: list[int]) -> list[int]
```

C returns a `malloc`'d array and writes its length to `*returnSize` (the caller frees it; for an empty input return `NULL` with `*returnSize = 0`).

### Hints

<details><summary>Show hints (open only if you are stuck)</summary>
<ol>
<li>3-way partition scans one index <code>i</code> through the region: smaller than pivot &rarr; swap to the front and advance; equal &rarr; advance; larger &rarr; swap to the back and <em>do not advance</em> (the swapped-in value still needs checking).</li>
<li>Randomness: use rejection sampling with <code>rand()</code> in C (see the reference answer; plain modulo can bias the index), <code>std::uniform_int_distribution</code> in C++, <code>Random.Next(lo, hi + 1)</code> in C#, <code>ThreadLocalRandom.current().nextInt(lo, hi + 1)</code> in Java, <code>random.randint(lo, hi)</code> in Python, <code>Math.floor(Math.random() * (hi - lo + 1)) + lo</code> in JS. Swap the chosen index with <code>nums[lo]</code> before partitioning.</li>
<li>Recurse on <code>[lo, lt-1]</code> and <code>[gt+1, hi]</code>, never on the equal band. Base case: a region of size 0 or 1. On 100,000 equal values, there should be one partition pass and no further recursive calls.</li>
</ol>
</details>

In [ ]:
# TODO: implement quick sort with a random pivot and a 3-way partition.
def quick_sort(nums: list[int]) -> list[int]:
    pass


In [ ]:
# Self-check — fails while quick_sort is still the untouched stub.
original = [5, 2, 4, 1, 3]
result = quick_sort(original)
assert result == [1, 2, 3, 4, 5], f"got {result}"
assert original == [5, 2, 4, 1, 3], "the input array must not be modified"

assert quick_sort([3, 1, 3, 3, 2, 3]) == [1, 2, 3, 3, 3, 3]
assert quick_sort([]) == [], "an empty array must be handled"

# All-equal input must stay fast: the 3-way partition skips it in one pass.
assert quick_sort([7] * 100_000) == [7] * 100_000

print("Problem 5 checks passed.")

## Problem 6 — Kth Largest Element (Quickselect)

**Difficulty:** Medium+ · **Topics:** Quickselect · Order statistics · Partition

**Optional exploration — not covered in class.** Try this independently if you want to explore order statistics beyond this session.

Given an integer array `nums` and an integer `k`, return the **k-th largest** element in sorted order — that is, the element at index `n - k` if the array were sorted ascending. `k` is 1-based, so `k = 1` asks for the maximum.

The expected solution is **quickselect**: partition like quicksort, but continue only in the region containing the target index. Use random pivots and 3-way partition for duplicates. Expected time O(n), with no need to sort everything.

Sorting the whole array also passes the tests, but it defeats the purpose. Duplicates matter here: `[3, 3, 3]` with `k = 2` must return `3`.

### Examples

**Example 1**
- Input: `nums = [3, 2, 1, 5, 6, 4], k = 2`
- Output: `5`
- Explanation: Sorted: [1, 2, 3, 4, 5, 6]. The second largest is 5.

**Example 2**
- Input: `nums = [3, 2, 3, 1, 2, 4, 5, 5, 6], k = 4`
- Output: `4`
- Explanation: Sorted: [1, 2, 2, 3, 3, 4, 5, 5, 6]. Index n - k = 5 holds 4.

### Constraints

- 1 ≤ `k` ≤ `nums.length` ≤ 100,000
- -10⁴ ≤ `nums[i]` ≤ 10⁴
- The array can contain many duplicate values.
- Do not use a built-in sort. O(n log n) passes, but the point is O(n) on average.

### Function signature (Python)

```python
def find_kth_largest(nums: list[int], k: int) -> int
```

The C reference answer may reorder the input. The other five reference answers preserve it by making a copy; C++ requires this because the parameter is a `const` reference. Selection itself is iterative.

### Hints

<details><summary>Show hints (open only if you are stuck)</summary>
<ol>
<li>Quickselect skips work: after 3-way partition, compare target index <code>n - k</code> with the equal band <code>[lt, gt]</code>. Return the pivot if the target is inside; otherwise keep only the smaller or larger region containing the target.</li>
<li>Why expected O(n)? A random pivot lands in the middle half with probability at least 1/2, leaving at most about 3/4 of the current region. A constant expected number of passes gives each such shrinkage, so the work sums geometrically. The region does not always halve.</li>
<li>Reuse the 3-way partition from Problem 5 and update <code>left</code>/<code>right</code> in a loop. Skip the entire equal band when narrowing the region. Test all-equal input and <code>k = 1</code>, <code>k = n</code>, and a middle rank.</li>
</ol>
</details>

In [ ]:
# TODO: implement find_kth_largest with iterative 3-way quickselect (expected O(n)).
def find_kth_largest(nums: list[int], k: int) -> int:
    pass


In [ ]:
# Self-check — fails while find_kth_largest is still the untouched stub.
assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
assert find_kth_largest([3, 3, 3], 2) == 3, "duplicates: [3, 3, 3] with k = 2 must return 3"

nums = [2, 1]
assert find_kth_largest(nums, 1) == 2
assert find_kth_largest(nums, 2) == 1

print("Problem 6 checks passed.")

## Where to go next

- [Recitation 1 page](https://harryc919.github.io/TA-CS301/recitation1.html) — full notes and reference solutions in six languages.
- [Recitation 0](https://harryc919.github.io/TA-CS301/recitation0.html) — environment setup and troubleshooting.
- [Introduction](https://harryc919.github.io/TA-CS301/intro.html) — office hours and course policies.

If a check cell reports a failure you believe is wrong, tell the TA.